## Operacions bàsiques

Un **tensor** és un array de *n* dimensions, com un `ndarray` de NumPy, amb tres propietats que farem servir constantment:

| Propietat | Què indica | Exemple |
| --- | --- | --- |
| `shape` | La mida de cada dimensió | `torch.Size([2, 2])` |
| `dtype` | El tipus dels elements | `torch.int64`, `torch.float32` |
| `device` | On viu el tensor | `cpu`, `cuda`, `mps` (GPU d'Apple) |

`torch.backends.mps.is_available()` comprova si podem fer servir la GPU d'un Mac amb Apple Silicon. Per triar el dispositiu de manera portable tenim `llms_from_scratch.get_device()`.

In [660]:
import torch
print(torch.backends.mps.is_available())

True


In [661]:
tensor2d = torch.tensor([[1, 2], [3, 4]])
print(tensor2d)

tensor([[1, 2],
        [3, 4]])


In [662]:
print(tensor2d.shape)
print(tensor2d.dtype)

torch.Size([2, 2])
torch.int64


PyTorch dedueix el `dtype` a partir de les dades: els enters són `int64` per defecte i els decimals, `float32`. Això té conseqüències:

- **Els pesos i les entrades d'un model han de ser decimals** (`float32`). Si li passes un tensor d'enters, `Linear` dona error. Es converteix amb `.float()` o `.to(torch.float32)`.
- **Les etiquetes de classe són enters** (`int64`). És el que espera `F.cross_entropy`, com veurem més avall.

`view` i `reshape` canvien la forma sense canviar les dades. La diferència és com tracten la memòria:

- **`view`** no copia mai: retorna una altra manera de llegir la *mateixa* memòria. Només funciona si el tensor és **contigu**, és a dir, si els seus elements estan seguits en memòria en l'ordre en què es llegeixen.
- **`reshape`** fa un `view` quan pot, i si no, copia les dades.

Per exemple, `tensor2d.T` no és contigu (la transposició només canvia l'ordre de lectura), així que `tensor2d.T.view(1, 4)` dona error, però `tensor2d.T.reshape(1, 4)` funciona.

In [663]:
# same result, but subtle differences in memory handling
print(tensor2d.view(1, 4))
print(tensor2d.reshape(1, 4))

tensor([[1, 2, 3, 4]])
tensor([[1, 2, 3, 4]])


In [664]:
print(tensor2d.T)

tensor([[1, 3],
        [2, 4]])


`matmul` és el producte matricial, que també es pot escriure `tensor2d @ tensor2d.T`. Multiplicar una matriu per la seva transposada dona tots els productes escalars entre les seves files:

$$
(AA^\top)_{ij} = \mathbf{a}_i \cdot \mathbf{a}_j
\qquad\Rightarrow\qquad
\begin{pmatrix} 1\cdot1+2\cdot2 & 1\cdot3+2\cdot4 \\ 3\cdot1+4\cdot2 & 3\cdot3+4\cdot4 \end{pmatrix}
= \begin{pmatrix} 5 & 11 \\ 11 & 25 \end{pmatrix}.
$$

Aquesta mateixa operació és el nucli de l'atenció del capítol 3: les puntuacions d'atenció són $QK^\top$, el producte escalar de cada *query* amb cada *key*.

In [665]:
print(tensor2d.matmul(tensor2d.T))

tensor([[ 5, 11],
        [11, 25]])


## Models com a grafs de càlcul

In [666]:
# this import is common in PyTorch code to prevent long lines of code when using activation functions like softmax, relu, etc.
import torch.nn.functional as F

### Regressió logística, logits i funció d'enllaç

En regressió logística binària, suposem que la variable resposta, condicionada a les covariables, segueix una distribució de Bernoulli:

$$
Y\mid X=x \sim \operatorname{Bernoulli}(p(x)),
\qquad p(x)=P(Y=1\mid X=x).
$$

Com que l'esperança d'una Bernoulli coincideix amb la seva probabilitat d'èxit, volem modelitzar:

$$
\mathbb{E}[Y\mid X=x]=p(x).
$$

El problema és que **una combinació lineal de les covariables pot prendre qualsevol valor real**, mentre que una probabilitat ha de pertànyer a $[0,1]$.

Per solucionar-ho, introduïm una **funció d'enllaç** $g$, i assumim que la probabilitat transformada és lineal en les covariables:

$$
\boxed{g(p(x))=\beta_0+\boldsymbol{\beta}^\top x=z}
$$

Aquesta és la **hipòtesi del model**: no suposem que la probabilitat sigui lineal, sinó que ho és una transformació d'aquesta probabilitat.

En regressió logística, escollim com a funció d'enllaç el **logit**:

$$
g(p)=\operatorname{logit}(p)=\log\left(\frac{p}{1-p}\right).
$$

La seva inversa és la **funció sigmoide**:

$$
g^{-1}(z)=\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Per tant, podem recuperar la probabilitat d'èxit a partir del predictor lineal:

$$
\boxed{p(x)=\sigma(\beta_0+\boldsymbol{\beta}^\top x)}
$$

**Correspondència amb el codi:**

- `z = x1 * w1 + b`: calcula el predictor lineal, que en regressió logística coincideix amb el *logit*.
- `a = torch.sigmoid(z)`: aplica la funció d'enllaç inversa per obtenir la probabilitat estimada.
- `loss = F.binary_cross_entropy(a, y)`: calcula la pèrdua mitjançant la log-versemblança negativa de la Bernoulli.

In [667]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

### Binary cross-entropy i màxima versemblança

La **binary cross-entropy (BCE)** és la **log-versemblança negativa** de la distribució de Bernoulli.

Partim de la seva funció de massa de probabilitat (PMF):

$$
P(Y=y\mid X=x)=p(x)^y(1-p(x))^{1-y}.
$$

Prenent el logaritme i canviant el signe, obtenim:

$$
\begin{aligned}
\mathcal{L}(y,p)
&=-\log P(Y=y\mid X=x)\\
&=-\log\left(p^y(1-p)^{1-y}\right)\\
&=\boxed{-\left[y\log p+(1-y)\log(1-p)\right]}.
\end{aligned}
$$

Recordem que la PMF de la Bernoulli es pot expressar en **forma de família exponencial**:

$$
P(Y=y;\theta)=\exp\left(y\theta-\log(1+e^\theta)\right),
$$

on el paràmetre natural és $\theta=\operatorname{logit}(p)$.

Com que en regressió logística utilitzem l'enllaç canònic, tenim:

$$
\theta=z=\beta_0+\boldsymbol{\beta}^\top x.
$$

Per tant, la mateixa pèrdua es pot expressar directament en termes del **logit**:

$$
\boxed{\mathcal{L}(y,z)=\log(1+e^z)-yz}.
$$

Així, la BCE en termes de probabilitats i la BCE en termes de logits són matemàticament equivalents.

Per a $n$ observacions independents, la pèrdua mitjana és:

$$
\mathcal{L}(\boldsymbol{\beta})
=-\frac{1}{n}\sum_{i=1}^{n}
\log P(Y=y_i\mid X=x_i;\boldsymbol{\beta}).
$$

**Minimitzar aquesta pèrdua equival a maximitzar la versemblança** i, per tant, a estimar els coeficients de la regressió logística per màxima versemblança, sempre que existeixi un màxim finit.

**Implementació en PyTorch:**

```python
# BCE a partir de probabilitats
p = torch.sigmoid(z)
loss = F.binary_cross_entropy(p, y)

# BCE directament a partir dels logits (preferible)
loss = F.binary_cross_entropy_with_logits(z, y)
```

La segona implementació és **numèricament més estable**, ja que combina la sigmoide i la cross-entropy sense haver de calcular explícitament els logaritmes de probabilitats properes a 0 o 1.

A més, la derivada de la pèrdua respecte del logit té una expressió especialment senzilla:

$$
\boxed{\frac{\partial\mathcal{L}}{\partial z}(y,z)=\frac{e^z}{1+e^z}-y=\sigma(z)-y=p-y}.
$$

És a dir, el gradient respecte del logit és simplement la diferència entre la probabilitat estimada i el valor observat.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/8.webp" width="600px">

In [668]:
import torch.nn.functional as F
from torch.autograd import grad

`autograd` construeix un graf de càlcul mentre fem operacions amb tensors que depenen de paràmetres amb `requires_grad=True`. Cada operació conserva la informació necessària per aplicar la regla de la cadena en sentit invers.

Aquí `w1` i `b` són els paràmetres; `z`, `a` i `loss` són resultats intermedis del graf. `grad(loss, w1)` retorna la derivada de la pèrdua respecte de `w1`, i `grad(loss, b)` fa el mateix amb el biaix. Les crides a `grad()` no omplen els atributs `.grad` dels paràmetres: això ho veurem amb `loss.backward()` a la cel·la següent.

In [669]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b 
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

# By default PyTorch destroys the computation graph after computing the gradients. 
# If you want to compute multiple gradients, you need to set retain_graph=True.
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [670]:
# more automatic way to compute gradients: .backward() computes the gradients of all the leaf nodes in the computation graph.
# the gradients are accumulated into the .grad attributes of the leaf nodes.
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


### `grad()` vs. `backward()`

Totes dues fan servir el graf de càlcul per aplicar la regla de la cadena, però retornen el resultat de manera diferent:

| | `grad(loss, w1)` | `loss.backward()` |
| --- | --- | --- |
| Sobre quins tensors | Els que li demanes | Totes les **fulles** del graf amb `requires_grad=True` |
| On deixa el resultat | El retorna | El **suma** a l'atribut `.grad` de cada fulla |
| Ús habitual | Inspeccionar un gradient concret | Bucles d'entrenament |

Una **fulla** és un tensor creat directament per nosaltres (`w1`, `b`), no el resultat d'una operació (`z`, `a`, `loss`). Hem posat `retain_graph=True` perquè, per defecte, PyTorch allibera el graf després de recórrer-lo. Sense aquest argument, el `loss.backward()` d'aquesta cel·la fallaria.

**Comprovació amb la derivada que hem deduït més amunt**, $\partial\mathcal{L}/\partial z = \sigma(z) - y$:

- $z = 1.1 \cdot 2.2 + 0 = 2.42$ i $\sigma(2.42) \approx 0.9183$, així que $\partial\mathcal{L}/\partial z \approx 0.9183 - 1 = -0.0817$.
- Com que $z = x_1 w_1 + b$, tenim $\partial z/\partial b = 1$, i per tant $\partial\mathcal{L}/\partial b = -0.0817$. ✔
- També $\partial z/\partial w_1 = x_1 = 1.1$, i per tant $\partial\mathcal{L}/\partial w_1 = 1.1 \cdot (-0.0817) \approx -0.0898$. ✔

## Xarxes neuronals multicapa

Un model de PyTorch és una subclasse de `torch.nn.Module`, amb dues parts:

- **`__init__`** crea les capes. Qualsevol `Module` que hi assignem com a atribut (aquí, `self.layers`) queda **registrat**, i els seus paràmetres apareixen automàticament a `model.parameters()`.
- **`forward`** descriu com es transforma l'entrada. No el cridem directament: fem `model(x)`, que crida `forward` i, a més, executa els *hooks* interns de PyTorch.

`Sequential` encadena capes: la sortida de cada una és l'entrada de la següent.

**Per què `ReLU` entre les capes `Linear`?** La composició de funcions lineals és lineal: $W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2)$. Sense una no-linealitat entremig, apilar capes seria equivalent a tenir-ne una de sola.

L'última capa no té activació. Retorna **logits**: puntuacions reals, no probabilitats, igual que la $z$ de la regressió logística.

In [671]:
# An example of a multilayer perceptron with two hidden layers
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        # sequential is a container that allows you to build a neural network by stacking layers on top of each other
        self.layers = torch.nn.Sequential(
            # non-linear activation functions are applied after each linear layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # number of output nodes of one hidden layer has to match the number of input nodes of the next hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, num_outputs)
        )
    def forward(self, x):
        # layers makes sense here because we don't have to call each layer individually in this forward method
        # logits because last layer is a linear layer and we haven't applied any activation function to it
        logits = self.layers(x)
        return logits

In [672]:
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [673]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 2213


Cada capa `Linear(n_in, n_out)` té una matriu de pesos de $n_{\text{out}} \times n_{\text{in}}$ i un vector de $n_{\text{out}}$ biaixos. `ReLU` no té paràmetres.

| Capa | Pesos | Biaixos | Total |
| --- | --- | --- | --- |
| `Linear(50, 30)` | 1500 | 30 | 1530 |
| `Linear(30, 20)` | 600 | 20 | 620 |
| `Linear(20, 3)` | 60 | 3 | 63 |
| **Total** | | | **2213** |

In [674]:
# weight matrix (parameters that are trainable if requires_grad=True)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0281,  0.1284,  0.1009,  ..., -0.0914, -0.1354,  0.1046],
        [ 0.1062, -0.0360,  0.0624,  ...,  0.0697, -0.0964,  0.0765],
        [-0.0045,  0.0584, -0.0003,  ..., -0.1287, -0.0371, -0.1072],
        ...,
        [-0.1057,  0.1387, -0.1316,  ...,  0.0019, -0.0163,  0.0589],
        [-0.1053, -0.0406, -0.1302,  ...,  0.0139, -0.0647, -0.0738],
        [-0.0511,  0.0880,  0.0709,  ...,  0.1037,  0.0555,  0.0037]],
       requires_grad=True)


In [675]:
# bias and weights are trainable by default
print(model.layers[0].bias)

Parameter containing:
tensor([ 0.0560, -0.0131,  0.1141,  0.0025,  0.0788,  0.1260,  0.0459,  0.0039,
        -0.0099, -0.1152, -0.1105,  0.0472,  0.0131,  0.0366, -0.1277,  0.0792,
        -0.0146,  0.0551, -0.0780,  0.0295,  0.0728, -0.1057,  0.0126, -0.0490,
         0.1161, -0.0296, -0.0827, -0.0906, -0.0129,  0.0642],
       requires_grad=True)


`Parameter containing:` indica que són objectes `nn.Parameter`: tensors amb `requires_grad=True` que un `Module` registra automàticament com a paràmetres seus.

La matriu de pesos de `Linear(50, 30)` té forma `(30, 50)`, és a dir, `(sortida, entrada)`, perquè la capa calcula $y = xW^\top + b$. Cada fila de $W$ conté els pesos d'una neurona de sortida.

Els pesos de les capes s'inicialitzen amb valors aleatoris petits. Si totes les neurones d'una mateixa capa comencessin amb els mateixos pesos, produirien la mateixa sortida i rebrien les mateixes actualitzacions: continuarien sent idèntiques durant l'entrenament. La inicialització aleatòria trenca aquesta simetria.

In [676]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


`torch.manual_seed(123)` fixa la llavor del generador aleatori. Així, cada vegada que executem aquesta cel·la obtenim els mateixos pesos inicials, i els resultats són reproduïbles.

Per defecte, `Linear` inicialitza els pesos amb una distribució uniforme a $\left[-\tfrac{1}{\sqrt{n_{\text{in}}}},\ \tfrac{1}{\sqrt{n_{\text{in}}}}\right]$. Amb $n_{\text{in}} = 50$, el límit és $1/\sqrt{50} \approx 0.14$, que és el rang dels valors que veiem.

In [677]:
torch.manual_seed(123)
X = torch.rand((1, 50))
# model(X) will call the forward method of the model, which will pass the input through the layers of the model and return the output
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


- **Forma de la sortida:** `X` té forma `(1, 50)`, o sigui un *batch* d'un sol exemple amb 50 característiques. Els models sempre treballen per batches, i la primera dimensió és la del batch. La sortida té forma `(1, 3)`: un logit per classe.
- **`grad_fn=<AddmmBackward0>`:** indica que `out` forma part d'un graf de càlcul, i que l'última operació ha estat `addmm` (*add + matrix multiply*, és a dir, $b + xW^\top$). PyTorch guarda aquest graf per poder fer `backward()` després.

Quan només volem predir, no cal construir el graf de càlcul. `torch.no_grad()` desactiva el seguiment de gradients dins del bloc: estalvia memòria i evita que la sortida tingui `grad_fn`.

`model.eval()` és una altra cosa: canvia el comportament de capes com `Dropout` o `BatchNorm` durant la inferència. En aquesta xarxa, formada només per `Linear` i `ReLU`, `eval()` no canvia els valors, però és la pràctica adequada quan avaluem un model.

In [678]:
with torch.no_grad():
    out = model(X)
    print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [679]:
# for a randomly initialized model without training, it is expected that the values are roughly equal
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
    print(out)

tensor([[0.3113, 0.3934, 0.2952]])


### Logits, softmax i *cross-entropy*

`softmax` converteix els logits en probabilitats que sumen 1:

$$
\operatorname{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_j e^{z_j}}.
$$

`dim=1` indica que normalitzem al llarg de la dimensió de les classes, no la del batch. És la generalització a $K$ classes de la sigmoide de la regressió logística.

**Durant l'entrenament no apliquem `softmax` al model.** `F.cross_entropy` rep directament els logits i aplica internament `log_softmax`, de la mateixa manera que `binary_cross_entropy_with_logits` és preferible a `sigmoid` + `binary_cross_entropy`: és numèricament més estable.

## Datasets i DataLoaders

Un problema de classificació de joguina: cada exemple té **2 característiques** i pertany a la **classe 0 o 1**. Les dues classes estan clarament separades (els exemples de la classe 0 tenen la primera coordenada negativa).

Les etiquetes són **índexs de classe** (`int64`), no vectors *one-hot*. És el format que espera `F.cross_entropy`.

In [680]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [681]:
from torch.utils.data import Dataset
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]        
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

Un `Dataset` només ha de saber fer dues coses:

- **`__getitem__(index)`** retorna l'exemple número `index`, com un parell `(característiques, etiqueta)`.
- **`__len__()`** retorna el nombre d'exemples.

El `DataLoader` farà servir aquests dos mètodes per construir els batches. Aquí les dades ja són tensors a memòria, però en un cas real `__getitem__` podria llegir de disc, tokenitzar text, etc. És el que farem al capítol 2 amb el text.

In [682]:
from torch.utils.data import DataLoader
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds, # train_ds is the dataset object that we created above, ToyDataset is a subclass of Dataset
    batch_size=2,
    shuffle=True,
    num_workers=0 # number of background processes to use for loading the data. 0 means that the data will be loaded in the main process, which is fine for small datasets
)
test_ds = ToyDataset(X_test, y_test)
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [683]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


El `DataLoader` agafa els índexs `0..len(dataset)-1`, els barreja (`shuffle=True`) i els agrupa de `batch_size` en `batch_size`. Per a cada grup crida `__getitem__` i apila els resultats en un sol tensor:

- **5 exemples amb `batch_size=2`** donen 3 batches, i l'últim només té 1 exemple.
- **`shuffle=True`** fa que l'ordre canviï a cada època. Així el model no aprèn res de l'ordre de les dades. Fem servir `shuffle=False` al conjunt de test perquè allà l'ordre no importa i volem resultats reproduïbles.
- **`torch.manual_seed(123)`** fa que la barreja també sigui reproduïble.

In [684]:
train_loader = DataLoader(
    dataset=train_ds, # train_ds is the dataset object that we created above, ToyDataset is a subclass of Dataset
    batch_size=2,
    shuffle=True,
    num_workers=0, # number of background processes to use for loading the data. 0 means that the data will be loaded in the main process, which is fine for small datasets
    drop_last=True 
)

Amb `drop_last=True`, el `DataLoader` descarta el darrer batch si no arriba a `batch_size`. Aquí tenim 5 exemples i batches de 2: obtenim **2 batches complets** i el cinquè exemple no es fa servir en aquesta època.

Pot ser útil quan una capa o un càlcul necessita batches de mida fixa. No és una regla general per millorar la convergència: també perdem dades de cada època. Com que `shuffle=True`, l'exemple descartat pot canviar a la següent època. Al conjunt de test normalment mantenim tots els exemples amb `drop_last=False`.

In [685]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/11.webp" width="600px">

## Un bucle d'entrenament típic

In [686]:
import torch.nn.functional as F
torch.manual_seed(123)
model = NeuralNetwork(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

### Què guarda l'optimizer?

En crear l'optimizer li passem `model.parameters()`. Això **no copia** els pesos: retorna els mateixos tensors `weight` i `bias` que viuen dins de cada `Linear`. A partir d'aquí, el model i l'optimizer apunten als mateixos objectes.

```python
optimizer.param_groups[0]["params"][0] is model.layers[0].weight  # True
```

L'optimizer **no guarda ni els gradients ni el batch**:

- **Els gradients** viuen a l'atribut `.grad` de cada paràmetre.
- **El batch** només hi arriba de manera indirecta, a través d'aquest `.grad` (ho veiem a sota del bucle).

Amb SGD, l'optimizer només recorda la llista de paràmetres i el *learning rate*. Altres optimitzadors, com Adam, també guarden un estat per paràmetre: mitjanes mòbils dels gradients anteriors.

In [687]:
num_epochs = 3
for epoch in range(num_epochs):
    model.train() # set the model to training mode
    for batch_idx, (features, labels) in enumerate(train_loader):
        # recordem que la última capa no té activació a la sortida
        logits = model(features)
        
        loss = F.cross_entropy(logits, labels) # Loss function
        
        optimizer.zero_grad() # clear the gradients of all optimized tensors from the previous batch
        loss.backward() # compute the gradients of the loss with respect to the model parameters
        optimizer.step() # update the model parameters using the computed gradients
    
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
                f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
                f" | Train/Val Loss: {loss:.2f}")
    
    model.eval()

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


### Què passa en cada pas del bucle?

| Línia | Què fa | Sobre quins tensors |
| --- | --- | --- |
| `logits = model(features)` | Calcula la sortida i construeix el **graf de càlcul d'aquest batch** | — |
| `loss = F.cross_entropy(...)` | Afegeix la pèrdua al final del graf | — |
| `optimizer.zero_grad()` | Posa `p.grad = None` | Els paràmetres que vam passar en crear l'optimizer |
| `loss.backward()` | Recorre el graf cap enrere i **suma** $\partial \mathcal{L} / \partial p$ a `p.grad` | Els tensors amb `requires_grad=True` que han intervingut en la `loss` |
| `optimizer.step()` | Actualitza cada paràmetre *in place*: $p \leftarrow p - \eta \cdot$ `p.grad` | Els paràmetres que vam passar en crear l'optimizer |

`zero_grad()` i `step()` no reben cap argument perquè ja saben sobre quins tensors han de treballar: els que van rebre en crear l'optimizer. `backward()` tampoc necessita saber res de l'optimizer: escriu els gradients directament dins de cada paràmetre.

### Com sap l'optimizer de quin batch parlem?

**No ho sap.** El batch arriba a l'optimizer a través de `.grad`:

1. `model(features)` i `F.cross_entropy(...)` construeixen un graf que només conté **aquest batch**.
2. `loss.backward()` escriu a `p.grad` el gradient de **aquesta** `loss`.
3. `optimizer.step()` llegeix el que hi hagi a `p.grad` en aquell moment.

Per això importa l'ordre de les crides:

- **Sense `zero_grad()`:** `backward()` *suma* al gradient anterior. Actualitzaríem amb la suma dels gradients d'aquest batch i de tots els anteriors.
- **Amb `step()` abans de `backward()`:** actualitzaríem amb els gradients del batch anterior, o amb res al primer pas.

Que `backward()` sumi en lloc de sobreescriure és expressament així. Permet l'**acumulació de gradients**, que es fa servir quan el batch que volem no cap a la memòria:

```python
optimizer.zero_grad()
for micro_x, micro_y in micro_batches:      # trossos d'un batch gran
    loss = F.cross_entropy(model(micro_x), micro_y) / len(micro_batches)
    loss.backward()                         # els gradients se sumen a p.grad
optimizer.step()                            # un sol pas, equivalent al batch gran
```

Per comprovar-ho pas a pas:

```python
w = model.layers[0].weight
print(w.grad)                     # None després de zero_grad()
loss = F.cross_entropy(model(features), labels)
loss.backward()
print(w.grad[0])                  # gradient d'aquest batch

before = w.detach().clone()
optimizer.step()
print(torch.allclose(w, before - 0.5 * w.grad))  # True: p -= lr * p.grad
```

In [688]:
# number of trainable parameters in the model
sum(p.numel() for p in model.parameters() if p.requires_grad) 

752

In [689]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [690]:
torch.set_printoptions(sci_mode=False)
probs = torch.softmax(outputs, dim=1)
print(probs)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [691]:
predictions = torch.argmax(probs, dim=1)
print(predictions)
print(predictions == y_train)

tensor([0, 0, 0, 1, 1])
tensor([True, True, True, True, True])


### De logits a prediccions

Després d'entrenar, `model.eval()` activa el mode d'avaluació i `torch.no_grad()` evita construir un graf que no necessitem. `outputs` té forma `(5, 2)`: una fila per exemple d'entrenament i una columna per classe.

`softmax(outputs, dim=1)` transforma cada fila en probabilitats que sumen 1. `argmax(..., dim=1)` tria l'índex de la classe més probable. Si només volem la classe predita, podem fer `torch.argmax(outputs, dim=1)` directament: `softmax` conserva l'ordre dels logits. La comparació `predictions == y_train` retorna un booleà per exemple; és una comprovació sobre dades d'entrenament, no una mesura de generalització.

In [692]:
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

In [693]:
print(compute_accuracy(model, train_loader))

1.0


### Exactitud del model

`compute_accuracy` recorre els batches del `DataLoader`, compta les prediccions correctes i divideix pel nombre total d'exemples que ha vist. Dins de la funció fem `argmax` directament sobre els logits perquè dona la mateixa classe que fer-lo sobre `softmax(logits)`. El resultat és un nombre entre 0 i 1.

Aquí passem `train_loader`, que té `drop_last=True`: en cada recorregut només s'avaluen **4 dels 5 exemples** d'entrenament. Per obtenir l'exactitud sobre tots els exemples, cal un `DataLoader` amb `drop_last=False`. També podem passar `test_loader` per avaluar dades que no s'han utilitzat per ajustar els pesos; amb només dos exemples, aquesta xifra seria molt poc estable.

La funció assumeix que les dades i el model són al mateix `device`. Si movem el model a la GPU més endavant, també haurem de moure-hi `features` i `labels` abans de comparar-los amb les prediccions.

## Desar i carregar models

In [694]:
# save
torch.save(model.state_dict(), "files/model.pth")
# and restore
model = NeuralNetwork(2, 2) # we need an instance of the model to load the state dict into
model.load_state_dict(torch.load("files/model.pth"))

<All keys matched successfully>

### Què es desa?

`model.state_dict()` és un diccionari amb els pesos i biaixos del model (i els buffers, si n'hi ha). `torch.save` l'escriu a `files/model.pth`. Aquesta ruta és relativa al directori des d'on s'executa el notebook, i la carpeta `files/` ha d'existir.

Per recuperar-lo, primer creem una altra instància de `NeuralNetwork(2, 2)` amb la mateixa arquitectura; després `load_state_dict` hi copia els valors desats. El fitxer no conté la definició de la classe ni l'estat de l'optimizer. Si només volem fer prediccions amb el model carregat, cal cridar `model.eval()`.

## Entrenar amb la GPU

In [695]:
print(torch.backends.mps.is_available())
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

True


In [696]:
tensor_1 = torch.tensor([1.0, 2.0, 3.0], device=device)
tensor_2 = torch.tensor([4.0, 5.0, 6.0], device=device)
print(tensor_1 + tensor_2)
# print(tensor_1 + tensor_2.to("cpu")) # RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

tensor([5., 7., 9.], device='mps:0')


### Model i tensors al mateix dispositiu

`mps` és l'accelerador de PyTorch per a GPU d'Apple. La selecció de `device` fa servir `mps` si està disponible i `cpu` en cas contrari. Els dos tensors de l'exemple es creen al mateix dispositiu i per això es poden sumar. En una operació entre tensors de dispositius diferents, PyTorch dona error; primer cal moure'n un amb `.to(device)`.

Quan fem `NeuralNetwork(2, 2).to(device)`, movem els paràmetres del model. Dins del bucle també movem **cada batch** (`features` i `labels`) al mateix dispositiu abans de calcular `logits` i la pèrdua. Creem l'optimizer després de moure el model, de manera que referencia els paràmetres que entrenarem.

Aquest exemple tria entre MPS i CPU perquè està pensat per a un Mac. En una màquina amb CUDA caldria incloure `cuda` en la selecció del dispositiu.

In [697]:
torch.manual_seed(123)
model = NeuralNetwork(2, 2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3

In [698]:
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features = features.to(device)
        labels = labels.to(device)
        
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
                f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
                f" | Train/Val Loss: {loss:.2f}")

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44


Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


El bucle d'entrenament és el mateix que a la CPU: `forward`, càlcul de la pèrdua, `zero_grad()`, `backward()` i `step()`. El canvi és la transferència del batch al `device` abans del `forward`. `model.train()` activa el mode d'entrenament a l'inici de cada època.

Si després volem avaluar aquest model, haurem de posar-lo en mode `eval()` i moure les dades d'avaluació al mateix dispositiu. Per imprimir o guardar prediccions com a arrays de NumPy, primer caldrà passar-les a la CPU amb `.cpu()` (i separar-les del graf amb `.detach()` si en tenen).